# 11: Benchmarks

smelt CPU vs HF float CPU vs HF float16 GPU vs onebitllms GPU vs llama.cpp CPU.

BitNet b1.58 2B. Measures tok/s, peak memory, kernel times.

In [1]:
import os
import sys

if "COLAB_GPU" in os.environ:
    os.system("pip install -q git+https://github.com/PritRaj1/smelt.git")
    os.system(
        "pip install -q 'transformers>=4.51,<5' psutil"
        " llama-cpp-python triton onebitllms matplotlib"
    )
else:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

import gc
import platform
import time

import matplotlib.pyplot as plt
import psutil
import torch
import torch.utils.benchmark as bench
from transformers import AutoModelForCausalLM, AutoTokenizer

import smelt

In [2]:
MODEL_ID = "microsoft/bitnet-b1.58-2B-4T"
PROMPT = "The meaning of life is"
MAX_NEW_TOKENS = 64
WARMUP = 8

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
ids = tokenizer.encode(PROMPT, return_tensors="pt")
proc = psutil.Process()

cpu_name = platform.processor() or "CPU"
gpu_label = torch.cuda.get_device_name(0)
n_threads = psutil.cpu_count(logical=False)
torch.set_num_threads(n_threads)

print(f"CPU: {cpu_name} ({n_threads} cores)")
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"GPU: {gpu_label}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

R = {}  # results accumulator

CPU: CPU (6 cores)
RAM: 16.4 GB
GPU: NVIDIA GeForce GTX 1650
VRAM: 3.9 GB


In [3]:
def bench_generate(model, input_ids, label):
    """Warmup + timed. Returns tok/s."""
    with torch.no_grad():
        model.generate(input_ids, max_new_tokens=WARMUP, do_sample=False)

    if input_ids.is_cuda:
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

    if input_ids.is_cuda:
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0
    n = out.shape[1] - input_ids.shape[1]
    tps = n / elapsed

    print(f"{label}: {tps:.1f} tok/s ({n} tokens in {elapsed:.2f}s)")
    return tps

## 1. smelt CPU

In [4]:
gc.collect()
rss0 = proc.memory_info().rss
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
model.eval()
smelt.quantize(model)
mem = (proc.memory_info().rss - rss0) / 1e6

R[f"smelt\n({cpu_name})"] = (bench_generate(model, ids, "smelt CPU"), mem)
del model
gc.collect()

`torch_dtype` is deprecated! Use `dtype` instead!
You have loaded a BitNet model on CPU and have a CUDA device available, make sure to set your model on a GPU device in order to run your model.


Loading weights:   0%|          | 0/542 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 2. HF float32 CPU

In [ ]:
rss0 = proc.memory_info().rss
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
model.eval()
mem = (proc.memory_info().rss - rss0) / 1e6

R[f"HF float32\n({cpu_name})"] = (bench_generate(model, ids, "HF float32 CPU"), mem)
del model
gc.collect()

## 3. HF float16 GPU

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
model.eval()
mem = torch.cuda.max_memory_allocated() / 1e6

ids_gpu = ids.to(model.device)
R[f"HF float16\n({gpu_label})"] = (bench_generate(model, ids_gpu, "HF float16 GPU"), mem)
del model
torch.cuda.empty_cache()
gc.collect()

## 4. onebitllms GPU

In [ ]:
from onebitllms import replace_linear_with_bitnet_linear

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
model = replace_linear_with_bitnet_linear(model)
model.eval()
mem = torch.cuda.max_memory_allocated() / 1e6

ids_gpu = ids.to(model.device)
R[f"onebitllms\n({gpu_label})"] = (bench_generate(model, ids_gpu, "onebitllms GPU"), mem)
del model
torch.cuda.empty_cache()
gc.collect()

## 5. llama.cpp CPU

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

gguf_path = hf_hub_download(
    repo_id="PritRaj1/bitnet-b1.58-2B-4T-GGUF",
    filename="bitnet-b1.58-2B-4T-TQ1_0.gguf",
)

gc.collect()
rss0 = proc.memory_info().rss
llm = Llama(model_path=gguf_path, n_ctx=256, verbose=False, n_threads=n_threads)
mem = (proc.memory_info().rss - rss0) / 1e6

llm(PROMPT, max_tokens=WARMUP)  # warmup
t0 = time.perf_counter()
out = llm(PROMPT, max_tokens=MAX_NEW_TOKENS)
elapsed = time.perf_counter() - t0
n = out["usage"]["completion_tokens"]
tps = n / elapsed
print(f"llama.cpp CPU: {tps:.1f} tok/s ({n} tokens in {elapsed:.2f}s)")

R[f"llama.cpp\n({cpu_name})"] = (tps, mem)
del llm
gc.collect()

## 6. Kernel microbenchmarks

In [ ]:
from smelt._clib import load_lib
from smelt.matmul import pack_tl1, quantize_ternary

lib = load_lib()

sizes = [
    ("attn_proj", 1, 2560, 2560),
    ("gate/up", 1, 2560, 6912),
    ("down", 1, 6912, 2560),
    ("lm_head", 1, 2560, 32768),
    ("prefill_16", 16, 2560, 2560),
    ("prefill_128", 128, 2560, 2560),
]

rows = []
for name, m, k, n in sizes:
    w_t, _ = quantize_ternary(torch.randint(-1, 2, (n, k), dtype=torch.float32))
    packed, n_pairs, n_padded = pack_tl1(w_t)
    x_i8 = torch.randint(-127, 128, (m, k), dtype=torch.int8)

    t_t = bench.Timer(
        stmt="lib.ternary_gemm(x, w, np, npairs)",
        globals={"lib": lib, "x": x_i8, "w": packed, "np": n_padded, "npairs": n_pairs},
    ).blocked_autorange(min_run_time=1.0)

    t_f = bench.Timer(
        stmt="torch.mm(x, w)",
        globals={"torch": torch, "x": torch.randn(m, k), "w": torch.randn(k, n)},
    ).blocked_autorange(min_run_time=1.0)

    rows.append((name, f"{m}x{k}x{n}", t_t.median * 1e6, t_f.median * 1e6))

print(f"{'name':<14} {'shape':<16} {'ternary us':>11} {'float us':>11} {'speedup':>8}")
print("-" * 64)
for name, shape, tt, tf in rows:
    print(f"{name:<14} {shape:<16} {tt:>11.1f} {tf:>11.1f} {tf / tt:>7.2f}x")

## 7. Barplots

In [ ]:
methods = list(R.keys())
speeds = [R[m][0] for m in methods]
mems = [R[m][1] for m in methods]

colors = []
for m in methods:
    if "smelt" in m.lower():
        colors.append("#e74c3c")

    elif "float16" in m.lower() or "onebit" in m.lower():
        colors.append("#2ecc71")

    else:
        colors.append("#3498db")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bars = ax1.barh(methods, speeds, color=colors)
for bar, v in zip(bars, speeds, strict=True):
    ax1.text(
        v + max(speeds) * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{v:.1f}",
        va="center",
        fontsize=10,
    )
ax1.set_xlabel("Tokens / second")
ax1.set_title(f"BitNet 2B generation ({MAX_NEW_TOKENS} tokens)")
ax1.invert_yaxis()

bars2 = ax2.barh(methods, mems, color=colors)
for bar, v in zip(bars2, mems, strict=True):
    ax2.text(
        v + max(mems) * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{v:.0f}",
        va="center",
        fontsize=10,
    )
ax2.set_xlabel("Memory (MB)")
ax2.set_title("Peak memory")
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig("benchmark.png", dpi=150, bbox_inches="tight")
plt.show()

# text summary
smelt_tps = R[methods[0]][0]
print(f"\n{'method':<28} {'tok/s':>8} {'mem MB':>8}")
print("-" * 46)
for m in methods:
    tps, mem = R[m]
    label = m.replace("\n", " ")
    print(f"{label:<28} {tps:>8.1f} {mem:>8.0f}")